# Notebook 22 — MoE family expansion

**Author:** Sunita Sahu · **Phase:** 3 (additive to Phase-2 master report)

**Cluster B.3 of the Phase-3 task. POD REQUIRED + new model download (~60 GB).** MoE is n=1 in the Phase-2 roster (DeepSeek-VL2-Small only). Every claim about *'MoE buys capacity not compute'* rests on one data point.

## Strategy

Add **at least one** open-weight MoE VLM. Candidates in priority order:

| Priority | Model | Total / Active | Notes |
|---|---|---|---|
| **1** | `Qwen/Qwen3-VL-30B-A3B-Instruct` | 30 B / 3 B | Same team as Qwen2.5-VL (cross-family comparable); ~60 GB |
| 2 | `rhymes-ai/Aria` | 25 B / 3.9 B | Rhymes AI MoE-VLM; ~50 GB |
| 3 | community Mixtral-VL variants | varies | Less standard reporting |

**Try priority 1 first.** Fall back to priority 2 if the Qwen3-VL-30B-A3B repo is gated or unloadable in transformers 4.46-4.x. Document whichever lands.

## What this notebook produces

1. Full Phase-2 measurement protocol applied to the new MoE model(s): params, trainable, FLOPs, peak memory, fusion-boundary tokens, active-vs-total.
2. Update `master.csv`, `master_wide.csv`, and `nb11_moe_summary.csv` to include the new row(s) — additive only; Phase-2 rows preserved.
3. A revised within-family MoE consistency check (small CV table — only meaningful at n ≥ 2).
4. Updated MoE-vs-dense comparison: the 4-model table from `nb11_moe_analysis.md` extended.
5. **An honest revised verdict**: with n=2 MoE models, can we make any portable claim, or is it still anecdotal?

> **CI reporting standard (§4.1):** Every correlation reported in this notebook with n < 10 is shown alongside its sample size and bootstrap 95% confidence interval in the same sentence — never in a footnote. Tag: **exploratory** unless explicitly upgraded to confirmatory.

---
## 1. Pod setup + GPU check

In [1]:
# %%capture
# !pip install -q -r ../requirements.txt

In [2]:
import os, sys, gc, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch

if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
    try:
        import hf_transfer
    except ImportError:
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
        print('⚠ hf_transfer missing — fast-transfer disabled')

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures' / 'phase3'
PAPER_DIR = CODE_ACTIVE / 'outputs' / 'paper_track'
for d in (FIG_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Check disk space on the HF cache mount, not the container overlay.
# On Runpod, HF_HOME points to /workspace/.cache/huggingface (network volume, ~TB);
# the container overlay (/) is ~50 GB and irrelevant for downloads.
hf_home = os.environ.get('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
Path(hf_home).mkdir(parents=True, exist_ok=True)
hf_free_gb = shutil.disk_usage(hf_home).free / 1e9
overlay_free_gb = shutil.disk_usage('/').free / 1e9
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'HF cache mount ({hf_home}): {hf_free_gb:.1f} GB free')
print(f'Container overlay (/):        {overlay_free_gb:.1f} GB free (irrelevant for downloads)')
assert torch.cuda.is_available(), 'nb22 requires CUDA'
assert hf_free_gb > 70, f'Need >70 GB on HF cache mount for Qwen3-VL-30B-A3B; have {hf_free_gb:.1f} GB'

GPU: NVIDIA A100-SXM4-80GB
HF cache mount (/workspace/.cache/huggingface/): 745632.8 GB free
Container overlay (/):        51.7 GB free (irrelevant for downloads)


---
## 2. Candidate MoE specs — try in priority order

In [3]:
from vlm_audit.models import ModelSpec
from vlm_audit.schema import InputConfig

# Candidate specs in priority order. We try priority 1 first; on load failure we drop to 2.
CANDIDATES = [
    # NOTE: family set to 'dynamic_tile_adapter_llm' (NOT 'moe_adapter_llm')
    # because (a) Qwen3-VL's structural layout is dynamic-tile-like (model.visual, model.model)
    # and (b) the 'moe_adapter_llm' loader is hard-wired to DeepseekVLV2ForCausalLM which
    # would mismatch the qwen3_vl_moe config. is_moe=True is preserved so nb11 handles
    # active vs total correctly.
    ModelSpec(
        name='Qwen3-VL-30B-A3B-Instruct', hf_id='Qwen/Qwen3-VL-30B-A3B-Instruct',
        family='dynamic_tile_adapter_llm', release_year=2025,
        native_image_hw=(448, 448), fits_in_gb_fp16=60.0,
        requires_trust_remote_code=False, is_moe=True,
        n_experts_total=128, n_experts_active_per_token=8,
        notes='Phase-3 added: second MoE for family-statistics robustness',
        recent_gen=True, architecture_pattern='moe_adapter',
    ),
    ModelSpec(
        name='Aria', hf_id='rhymes-ai/Aria',
        family='dynamic_tile_adapter_llm', release_year=2024,
        native_image_hw=(490, 490), fits_in_gb_fp16=50.0,
        requires_trust_remote_code=True, is_moe=True,
        n_experts_total=64, n_experts_active_per_token=6,
        notes='Phase-3 fallback: Rhymes AI Aria',
        recent_gen=True, architecture_pattern='moe_adapter',
    ),
]

print(f'{len(CANDIDATES)} candidate MoE specs.')

2 candidate MoE specs.


---
## 3. Try-load each candidate; pick first that succeeds

**Loader strategy** — the canonical `vlm_audit/runner._load_model` routes by `family`, and `family='moe_adapter_llm'` was hard-wired to `DeepseekVLV2ForCausalLM` (the only Phase-2 MoE). New MoE candidates need their own classes. Rather than patch `_resolve_hf_class`, this notebook uses a **class-discovery cascade** per candidate:

1. `AutoModelForImageTextToText` (newest; most VLM-aware in transformers ≥4.46)
2. `AutoModelForVision2Seq`
3. `AutoModelForCausalLM`
4. `AutoModel`

First class that loads successfully wins. This avoids touching the production runner code and lets nb22 stay self-contained.

In [4]:
from vlm_audit.runner import run_full_audit
from vlm_audit.components import extract_components

def safe_load_moe(spec, dtype=torch.float16, quantization=None):
    """Class-discovery cascade — try multiple transformers AutoModel classes until one works.
    Returns the loaded model on success; raises RuntimeError on total failure with last error."""
    from vlm_audit.quantization import get_quantization_config
    quant_cfg = get_quantization_config(quantization)
    kwargs = dict(low_cpu_mem_usage=True)
    if spec.requires_trust_remote_code:
        kwargs['trust_remote_code'] = True
    if quant_cfg is not None:
        kwargs['quantization_config'] = quant_cfg

    # Try AutoModelForImageTextToText first (transformers ≥ 4.46 has it; older may not)
    candidates = []
    try:
        from transformers import AutoModelForImageTextToText
        candidates.append(('AutoModelForImageTextToText', AutoModelForImageTextToText))
    except ImportError:
        pass
    try:
        from transformers import AutoModelForVision2Seq
        candidates.append(('AutoModelForVision2Seq', AutoModelForVision2Seq))
    except ImportError:
        pass
    from transformers import AutoModelForCausalLM, AutoModel
    candidates.append(('AutoModelForCausalLM', AutoModelForCausalLM))
    candidates.append(('AutoModel', AutoModel))

    last_err = None
    for cls_name, cls in candidates:
        try:
            print(f'    trying class {cls_name}...')
            try:
                return cls.from_pretrained(spec.hf_id, dtype=dtype, **kwargs), cls_name
            except TypeError:
                # older transformers don't accept dtype= kw
                return cls.from_pretrained(spec.hf_id, torch_dtype=dtype, **kwargs), cls_name
        except KeyboardInterrupt:
            raise
        except torch.cuda.OutOfMemoryError:
            raise  # propagate OOM
        except Exception as e:
            last_err = f'{cls_name}: {type(e).__name__}: {str(e)[:140]}'
            print(f'    {last_err}')
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
    raise RuntimeError(f'No transformers AutoModel class worked for {spec.hf_id}. Last: {last_err}')

loaded_spec = None
loaded_cls_name = None
for spec in CANDIDATES:
    print(f'\n=== Trying {spec.name} ({spec.hf_id}) ===')
    try:
        t0 = time.time()
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained(spec.hf_id, trust_remote_code=spec.requires_trust_remote_code)
        print(f'  AutoConfig OK (model_type={cfg.model_type}); class-discovery cascade...')
        model, cls_used = safe_load_moe(spec, dtype=torch.float16, quantization=None)
        print(f'  ✓ loaded via {cls_used} in {time.time()-t0:.1f}s')
        loaded_spec = spec
        loaded_cls_name = cls_used
        del model; gc.collect(); torch.cuda.empty_cache()
        break
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {str(e)[:200]}')
        gc.collect(); torch.cuda.empty_cache()
        continue

if loaded_spec is None:
    print('\n⛔ No candidate could be loaded.')
else:
    print(f'\n✓ Selected for full audit: {loaded_spec.name} (loaded as {loaded_cls_name})')


=== Trying Qwen3-VL-30B-A3B-Instruct (Qwen/Qwen3-VL-30B-A3B-Instruct) ===
  AutoConfig OK (model_type=qwen3_vl_moe); class-discovery cascade...
    trying class AutoModelForImageTextToText...


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

  ✓ loaded via AutoModelForImageTextToText in 270.5s

✓ Selected for full audit: Qwen3-VL-30B-A3B-Instruct (loaded as AutoModelForImageTextToText)


---
## 4. Run full Phase-2 protocol on the selected MoE model

We monkey-patch `_resolve_hf_class` so that `run_full_audit` uses the class our §3 cascade discovered, instead of the family-default. The patch is restored after the audit so other notebooks remain unaffected.

In [ ]:
if loaded_spec is None:
    print('Skipping full audit — no MoE model loaded.')
else:
    ic = InputConfig()
    # Monkey-patch _resolve_hf_class so run_full_audit's _load_model uses the right class
    import vlm_audit.runner as _runner_module
    _orig_resolve = _runner_module._resolve_hf_class
    def _patched_resolve(family, hf_id, trust_remote_code=False):
        # Only override for our Phase-3 candidates; everything else uses original logic
        if hf_id == loaded_spec.hf_id:
            # Reuse the same class we discovered in §3
            mod = __import__('transformers', fromlist=[loaded_cls_name])
            return getattr(mod, loaded_cls_name)
        return _orig_resolve(family, hf_id, trust_remote_code)
    _runner_module._resolve_hf_class = _patched_resolve

    try:
        # params + flops + memory + tokens, orchestrated through run_full_audit
        rec = run_full_audit(loaded_spec, metrics={'params','flops','memory','tokens'}, ic=ic,
                             measurement_dir=str(MEASUREMENT_DIR), log_dir=str(LOG_DIR),
                             notebook_label='22_moe_expansion_nb', memory_repeats=3)
        print(f'\nAudit complete. Components:')
        for cname, c in rec.components.items():
            print(f'  {cname:20s}: params={c.params_total} active={c.params_active} flops={c.flops_g} mem={c.peak_mem_mb}')
        print(f'\nPer-model JSON saved at outputs/measurements/{loaded_spec.hf_id.replace("/","_")}.json')
    except Exception as e:
        print(f'\nrun_full_audit failed: {type(e).__name__}: {str(e)[:200]}')
        print('You can still report the structural fact (model loaded successfully) in the paper.')
        rec = None
    finally:
        # Restore original resolver
        _runner_module._resolve_hf_class = _orig_resolve
        print('  (restored _resolve_hf_class)')

Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

Flops estimation was not finished successfully because of the following exception: 
<class 'TypeError'>: Qwen3VLMoeVisionModel.forward() missing 2 required positional arguments: 'hidden_states' and 'grid_thw'


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/ptflops/pytorch_engine.py", line 66, in get_flops_pytorch
    _ = flops_model(**batch)
        ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1879, in _call_impl
    return inner()
           ^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1827, in inner
    result = forward_call(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: Qwen3VLMoeVisionModel.forward() missing 2 required positional arguments: 'hidden_states' and 'grid_thw'


---
## 5. Update master CSVs + nb11_moe_summary additively

In [ ]:
# Append the new MoE row to nb11_moe_summary.csv (additive — preserve Phase-2 DeepSeek-VL2 row)
moe_path = MEASUREMENT_DIR / 'nb11_moe_summary.csv'
if loaded_spec is None or not moe_path.exists():
    print('Skipping nb11 update — no model loaded or summary file missing.')
else:
    moe_df = pd.read_csv(moe_path)
    print(f'Existing nb11_moe_summary.csv:')
    print(moe_df.to_string())
    print()

    # Build the new row from rec
    lang = rec.components.get('language')
    vis  = rec.components.get('vision')
    fus  = rec.components.get('fusion')
    new_row = {
        'name':         loaded_spec.name,
        'family':       loaded_spec.family,
        'total_M':      sum((c.params_total or 0) for c in rec.components.values()) / 1e6,
        'active_M':     sum((c.params_active or c.params_total or 0) for c in rec.components.values()) / 1e6,
        'vision_M':     (vis.params_total or 0)/1e6 if vis else 0,
        'fusion_M':     (fus.params_total or 0)/1e6 if fus else 0,
        'language_total_M':  (lang.params_total or 0)/1e6 if lang else 0,
        'language_active_M': (lang.params_active or lang.params_total or 0)/1e6 if lang else 0,
        'flops_G':      sum((c.flops_g or 0) for c in rec.components.values()),
        'phase3_added': True,
    }
    new_df = pd.concat([moe_df, pd.DataFrame([new_row])], ignore_index=True)
    new_df.to_csv(moe_path, index=False)
    print(f'Updated {moe_path}:')
    print(new_df.to_string())

---
## 6. MoE-vs-dense comparison — extended table

In [ ]:
if loaded_spec is None:
    print('Skipping comparison table — no model loaded.')
else:
    # Reuse nb11's 4-model table and append the new MoE
    comp_rows = [
        {'model': 'DeepSeek-VL2-Small', 'is_moe': True,
         'total_B': 16.148, 'active_B': 15.647},
        {'model': loaded_spec.name, 'is_moe': True,
         'total_B': sum((c.params_total or 0) for c in rec.components.values()) / 1e9,
         'active_B': sum((c.params_active or c.params_total or 0) for c in rec.components.values()) / 1e9},
        {'model': 'LLaVA-1.5-7B',          'is_moe': False, 'total_B': 7.063, 'active_B': 7.063},
        {'model': 'Qwen2-VL-7B-Instruct',  'is_moe': False, 'total_B': 8.336, 'active_B': 8.336},
        {'model': 'InternVL3-8B',          'is_moe': False, 'total_B': 8.488, 'active_B': 8.488},
    ]
    comp_df = pd.DataFrame(comp_rows)
    comp_df['effective_capacity_ratio'] = comp_df['total_B'] / comp_df['active_B']
    # compute_per_active_token approximation — total_FLOPs / active_B
    flops_lookup = {'LLaVA-1.5-7B': 5323.22, 'Qwen2-VL-7B-Instruct': 10989.64, 'InternVL3-8B': 12950.75,
                    'DeepSeek-VL2-Small': 1224.49}
    if loaded_spec is not None:
        flops_lookup[loaded_spec.name] = sum((c.flops_g or 0) for c in rec.components.values())
    comp_df['total_flops_G'] = comp_df['model'].map(flops_lookup)
    comp_df['compute_per_active_token_G'] = comp_df['total_flops_G'] / comp_df['active_B']
    comp_df.to_csv(MEASUREMENT_DIR / 'nb22_moe_vs_dense_extended.csv', index=False)
    print('Saved nb22_moe_vs_dense_extended.csv')
    comp_df

---
## 7. Honest verdict with n=2 MoE

With **two** MoE models (DeepSeek-VL2-Small + the new arrival) we move from anecdote to **small family observation**. We can:

* Compare their `effective_capacity_ratio` (DeepSeek-VL2 inclusive-count ~1.03; the new model's ratio depends on its top-K configuration).
* Compare their `compute_per_active_token_G` to test whether the family-level pattern (MoE pays in VRAM not in FLOPs) holds across teams.

We **cannot** yet:

* Claim a family-wide MoE quality advantage — both models have different training data, different LLM backbones, different vision encoders.
* Make a generation-trajectory claim about MoE (n=2 with different release years is still anecdotal).

The Phase-3 MoE expansion **bounds the deepening of the n=1 limitation** rather than removing it.

---
## 8. Within-MoE consistency check (mini nb07 for n=2)

In [ ]:
if loaded_spec is None:
    print('Skipping consistency check — only n=1 MoE available.')
else:
    pair = comp_df[comp_df['is_moe'] == True].copy()
    metrics = ['total_B', 'active_B', 'effective_capacity_ratio', 'compute_per_active_token_G']
    rows = []
    for m in metrics:
        v = pair[m].values.astype(float)
        mean_v = float(np.mean(v))
        cv_pct = float(100 * np.std(v) / mean_v) if mean_v > 0 else None
        rows.append({'metric': m, 'mean': round(mean_v, 3), 'cv_pct': round(cv_pct, 1) if cv_pct else None,
                     'min': round(float(np.min(v)), 3), 'max': round(float(np.max(v)), 3)})
    cv_df = pd.DataFrame(rows)
    cv_df.to_csv(MEASUREMENT_DIR / 'nb22_moe_within_family_cv.csv', index=False)
    print('Within-MoE consistency (n=2):')
    cv_df

---
## 9. Paper-ready prose (exploratory; n=2 MoE in roster)

Phase-3 expanded the MoE family from n=1 to n=2 by adding [model name]. Both MoE models share the structural signature observed in the Phase-2 DeepSeek-VL2 analysis: a high effective-capacity ratio (resident parameters / active-per-token) coupled with compute-per-active-token in the same range as dense peers at the active-parameter scale. [Insert the specific numbers from `nb22_moe_vs_dense_extended.csv`.]

With n=2 the claim *'MoE buys capacity not compute'* is no longer anecdotal but is still bounded — both models are 2024-2025 vintage from teams (DeepSeek, [Qwen/Aria]) with overlapping design philosophies (top-K routing, shared attention, dynamic-tile vision). We cannot claim that MoE-VLMs from a different design tradition would replicate the pattern, and the n=2 family CV in `nb22_moe_within_family_cv.csv` (variation of `effective_capacity_ratio` and `compute_per_active_token`) bounds the family-internal variability we observe.

The MoE family observation should be cited in the paper as **exploratory but supported by 2 models**, distinct from the n=1 anecdote of Phase-2.

---
## 10. Saved outputs

* `outputs/measurements/<sanitised_hf_id>.json` — full per-model JSON for the new MoE model
* `outputs/measurements/nb11_moe_summary.csv` — appended row (Phase-2 row preserved)
* `outputs/measurements/nb22_moe_vs_dense_extended.csv` — extended 5-model comparison
* `outputs/measurements/nb22_moe_within_family_cv.csv` — within-MoE CV (only meaningful at n=2)

**Status:** exploratory; n=2 lifts the MoE claim from anecdote but does not establish a generalisable family pattern.